<a href="https://colab.research.google.com/github/opherdonchin/BayesShortCourse/blob/main/sleep/05_varying_intercept_slope.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sleep deprivation 5 — Varying intercepts and slopes

Notebook 4 introduced a hierarchical model in detail. Here the full model is supplied up front.

The new task is to **read a hierarchical model**, run the familiar Bayesian workflow, and identify which posterior quantities answer particular questions.

## Setup

This course pins PyMC and the modular ArviZ packages for reproducibility because their APIs can change across major versions.

In [ ]:
%pip install -q \
    "pandas==2.2.3" \
    "pymc==6.3.2" \
    "arviz-base==1.3.0" \
    "arviz-stats==1.3.2" \
    "arviz-plots[matplotlib]==1.3.1"

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
import pymc as pm
import arviz_base as azb
import arviz_plots as azp
import arviz_stats as azs

RANDOM_SEED = 20260924
azp.style.use("arviz-variat")

print("PyMC:", pm.__version__)
print("arviz-base:", azb.__version__)
print("arviz-plots:", azp.__version__)
print("arviz-stats:", azs.__version__)

## Data

The original study contains two adaptation/training days followed by a baseline measurement and then seven nights of severe sleep restriction. You can read about the original study here: [Belenky J Sleep Res. 2003](https://doi.org/10.1046/j.1365-2869.2003.00337.x)

Following the chapter, we drop original days 0–1 and subtract 2 from the remaining day number. Therefore **`Days = 0` is the baseline measurement before sleep deprivation begins**.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/lme4/sleepstudy.csv"

sleep = pd.read_csv(DATA_URL).drop(columns="rownames")
sleep = sleep.loc[sleep["Days"] >= 2].copy()
sleep["Days"] = sleep["Days"] - 2
sleep["Subject"] = sleep["Subject"].astype(str)
sleep = sleep.reset_index(drop=True)

participants = sorted(sleep["Subject"].unique(), key=int)
participant_to_idx = {participant: i for i, participant in enumerate(participants)}
participant_idx = sleep["Subject"].map(participant_to_idx).to_numpy()

assert participant_idx.min() == 0
assert participant_idx.max() == len(participants) - 1
assert np.array_equal(
    np.asarray(participants)[participant_idx],
    sleep["Subject"].to_numpy(),
)

print(f"{len(participants)} participants, {len(sleep)} observations")
print(f"Days: {sleep['Days'].min()} to {sleep['Days'].max()}")
sleep.head()

### Plotting helper

The participant plotting helper is supplied. It can optionally restrict plots to selected participants using the standard ArviZ `coords` argument.

In [ ]:
LM_VISUALS = {
    "pe_line": {"color": "C1"},
    "ci_band": {"color": "C0"},
    "observed_scatter": {"color": "black", "alpha": 1, "zorder": 3, "s": 10},
}

PARTICIPANT_DAY = xr.Coordinates.from_pandas_multiindex(
    pd.MultiIndex.from_frame(
        sleep[["Subject", "Days"]],
        names=["participant", "day"],
    ),
    "obs_id",
)

def plot_participants(dt, group, var, coords=None):
    """One panel per participant, optionally restricted with ArviZ coords."""
    def reshape(ds):
        return ds.assign_coords(PARTICIPANT_DAY).unstack("obs_id")

    panels = xr.DataTree.from_dict({
        group: reshape(dt[group].to_dataset()),
        "observed_data": reshape(dt["observed_data"].to_dataset()),
        "constant_data": reshape(dt["constant_data"].to_dataset()),
    })

    pc = azp.plot_lm(
        panels,
        x="days",
        y=var,
        y_obs="y",
        group=group,
        plot_dim="day",
        coords=coords,
        ci_prob=(0.50, 0.90),
        ci_kind="hdi",
        point_estimate="mean",
        smooth=False,
        cols=["participant"],
        col_wrap=6,
        figure_kwargs={"figsize": (11, 5.5), "sharex": True, "sharey": True},
        visuals={**LM_VISUALS, "xlabel": False, "ylabel": False},
    )

    pc.add_legend("prob", title="HDI")
    fig = pc.get_viz("figure")
    fig.supxlabel("Days of sleep deprivation")
    fig.supylabel("Reaction time (ms)")
    return pc

## 1. Read the varying-intercept, varying-slope model

### 1.1 The model

The full model is supplied:

$$
y_i \sim \operatorname{Normal}(\mu_{y,i}, sd_y)
$$

$$
\mu_{y,i}
=
b_{0,s[i]}
+
b_{1,s[i]}\,\mathrm{days}_i
$$

$$
b_{0,s}
\sim
\operatorname{Normal}(\mu_{b0}, sd_{b0})
$$

$$
b_{1,s}
\sim
\operatorname{Normal}(\mu_{b1}, sd_{b1}).
$$

The intercept and slope hierarchies are independent in this short-course model. This is a deliberate simplification: it allows participant baselines and sleep-deprivation effects to vary, but it does not estimate whether those two characteristics are correlated across participants.

### 1.2 The complete PyMC implementation

The intercept hierarchy is the same one introduced in Notebook 4. The new slope hierarchy follows the same pattern: a population center, a between-participant standard deviation, and one partially pooled slope per participant.

In [ ]:
coords = {
    "obs_id": np.arange(len(sleep)),
    "participant": participants,
}

# Hyperprior constants for the intercept hierarchy
mu_mu_b0 = 250
sd_mu_b0 = 100
sd_sd_b0 = 25

# Hyperprior constants for the slope hierarchy
mu_mu_b1 = 0
sd_mu_b1 = 20
sd_sd_b1 = 10

# Prior constant for observation-level variation
mu_sd_y = 50

with pm.Model(coords=coords) as model:
    days = pm.Data("days", sleep["Days"].to_numpy(), dims="obs_id")
    pidx = pm.Data("participant_idx", participant_idx, dims="obs_id")

    # Varying intercepts
    mu_b0 = pm.Normal("mu_b0", mu=mu_mu_b0, sigma=sd_mu_b0)
    sd_b0 = pm.Exponential("sd_b0", scale=sd_sd_b0)
    b0 = pm.Normal("b0", mu=mu_b0, sigma=sd_b0, dims="participant")

    # Varying slopes
    mu_b1 = pm.Normal("mu_b1", mu=mu_mu_b1, sigma=sd_mu_b1)
    sd_b1 = pm.Exponential("sd_b1", scale=sd_sd_b1)
    b1 = pm.Normal(
        "b1",
        mu=mu_b1,
        sigma=sd_b1,
        dims="participant",
    )

    # Useful derived quantity: expected change across all seven days
    change_7 = pm.Deterministic(
        "change_7",
        7 * b1,
        dims="participant",
    )

    # Likelihood
    sd_y = pm.Exponential("sd_y", scale=mu_sd_y)
    mu_y = pm.Deterministic(
        "mu_y",
        b0[pidx] + b1[pidx] * days,
        dims="obs_id",
    )
    y = pm.Normal(
        "y",
        mu=mu_y,
        sigma=sd_y,
        observed=sleep["Reaction"].to_numpy(),
        dims="obs_id",
    )

print(model)

In [ ]:
pm.model_to_graphviz(model)

### 1.3 Which quantities vary by participant?

Which model variables contain one value for every participant?

- answer here

### 1.4 Which parameter is the population-average sleep-deprivation slope?

Name the parameter that represents the average change in expected reaction time per day across participants.

- answer here

### 1.5 Which parameters describe the population distribution of slopes?

Which parameter gives its center, and which gives its between-participant scale?

- answer here

### 1.6 What controls the size of participant slope differences?

Which parameter controls how widely participant slopes vary around the population mean?

- answer here

### 1.7 What exchangeability assumption does the slope hierarchy make?

What does the model assume about the relationship between participant labels and their slopes before seeing the data?

- answer here

### 1.8 Must `mu_b1` equal the average of these 18 participant slopes?

Distinguish the center of the population distribution from the realized average of this finite sample of participants.

- answer here

### 1.9 Which participant-specific quantities determine `mu_y`?

Name the two participant-specific regression parameters selected by `participant_idx`.

- answer here

### 1.10 What is the difference between `mu_y` and `y`?

Which one is expected reaction time and which one includes observation-level variation?

- answer here

### 1.11 What does `change_7` represent?

Give its units and interpretation.

- answer here

## 2. Check the prior implications

### 2.1 Criteria

The slope hierarchy should:

- allow a broad range of plausible population-average sleep-deprivation effects;
- allow meaningful participant-to-participant variation in slopes;
- avoid making enormous seven-day changes routine.

The full prior predictive distribution should also satisfy the general plausibility criteria established in the earlier notebooks.

### 2.2 Draw from the prior and inspect the new slope hierarchy.

In [ ]:
with model:
    prior = pm.sample_prior_predictive(
        draws=500,
        var_names=[
            "mu_b1",
            "sd_b1",
            "b1",
            "change_7",
            "mu_y",
            "y",
        ],
        random_seed=RANDOM_SEED,
    )

azp.plot_dist(
    prior,
    group="prior",
    var_names=["mu_b1", "sd_b1"],
    ci_prob=0.90,
    point_estimate="mean",
);

azp.plot_forest(
    prior,
    group="prior",
    var_names=["change_7"],
    combined=True,
    ci_probs=(0.50, 0.90),
    figure_kwargs={"figsize": (7, 6)},
);

### 2.3 Are the slope priors reasonable?

Do the plotted priors allow meaningful differences in seven-day change without making implausibly enormous changes routine?

- answer here

### 2.4 Generate and plot prior predictive reaction times.

In [ ]:
plot_participants(prior, "prior_predictive", "y")
plt.show()

### 2.5 Do the prior predictive simulations satisfy the established criteria?

Judge support, baseline scale, changes across days, and participant-to-participant variation.

- answer here

## 3. Fit and diagnose the model

### 3.1 Sample from the posterior.

In [ ]:
with model:
    idata = pm.sample(
        draws=1000,
        tune=1500,
        chains=4,
        random_seed=RANDOM_SEED,
    )

### 3.2 Check population-level diagnostics.

In [ ]:
print("Divergences:", int(idata["sample_stats"]["diverging"].sum().item()))

population_summary = azs.summary(
    idata,
    var_names=["mu_b0", "sd_b0", "mu_b1", "sd_b1", "sd_y"],
    ci_prob=0.90,
    ci_kind="hdi",
    round_to=2,
)
population_summary

In [ ]:
azp.plot_trace_dist(
    idata,
    var_names=["mu_b0", "sd_b0", "mu_b1", "sd_b1", "sd_y"],
);

### 3.3 Do the population-level parameters meet the diagnostic criteria?

Use the same criteria established in Notebook 1: no divergences, R-hat close to 1, adequate bulk and tail ESS, and well-mixed traces.

- answer here

### 3.4 Inspect a small subset of participant effects.

In [ ]:
participant_diagnostics = azs.summary(
    idata,
    var_names=["b0", "b1"],
    kind="diagnostics",
    round_to=2,
)

display(pd.DataFrame(
    {
        "value": [
            participant_diagnostics["r_hat"].max(),
            participant_diagnostics["ess_bulk"].min(),
            participant_diagnostics["ess_tail"].min(),
        ]
    },
    index=["largest R-hat", "smallest bulk ESS", "smallest tail ESS"],
))

diagnostic_participants = [
    participants[0],
    participants[len(participants) // 2],
    participants[-1],
]
diagnostic_coords = {"participant": diagnostic_participants}

azs.summary(
    idata,
    var_names=["b0", "b1"],
    coords=diagnostic_coords,
    ci_prob=0.90,
    ci_kind="hdi",
    round_to=2,
)

In [ ]:
azp.plot_trace_dist(
    idata,
    var_names=["b0", "b1"],
    coords=diagnostic_coords,
);

### 3.5 Do the displayed participant-level parameters meet the same criteria?

- answer here

## 4. Examine the fitted slope hierarchy

### 4.1 Which parameter gives the population-average daily sleep-deprivation effect?

- answer here

### 4.2 Plot the posterior distribution of that parameter.

In [ ]:
# answer here

### 4.3 What range of population-average daily effects is credible?

Use the 90% HDI from the posterior summary.

- answer here

### 4.4 Which parameter controls participant-to-participant variation in slopes?

- answer here

### 4.5 Plot its posterior distribution.

In [ ]:
# answer here

### 4.6 What range of slope-variation scales is credible?

Use the 90% HDI for `sd_b1`.

- answer here

### 4.7 Which posterior variable contains the participant-specific slopes?

- answer here

### 4.8 Plot the participant-specific slopes.

In [ ]:
# answer here

### 4.9 Does the fitted model support meaningful heterogeneity in sleep-deprivation effects?

Base the answer on the participant intervals and the posterior for `sd_b1`.

- answer here

### 4.10 Which variable gives each participant's expected seven-day change?

- answer here

### 4.11 Plot those participant-specific seven-day changes.

In [ ]:
# answer here

## 5. Examine expected participant trajectories

### 5.1 Which variable represents expected reaction time without residual observation noise?

- answer here

### 5.2 Plot the posterior expected trajectories.

In [ ]:
# answer here

### 5.3 Can expected participant trajectories now differ in both baseline and rate of change?

- answer here

## 6. Posterior predictive check — participant trajectories

### 6.1 Generate and plot posterior predictive reaction times.

In [ ]:
with model:
    pm.sample_posterior_predictive(
        idata,
        var_names=["y"],
        extend_inferencedata=True,
        random_seed=RANDOM_SEED,
    )

plot_participants(idata, "posterior_predictive", "y")
plt.show()

### 6.2 What is added when we move from `mu_y` to posterior predictive `y`?

- answer here

### 6.3 Does the model reproduce participant-to-participant differences in trajectories?

Check both overall level and change across days.

- answer here

## 7. Posterior predictive check — overall distribution

### 7.1 Why make another posterior predictive plot?

The participant panels primarily check the conditional trajectories.

We now pool all observations and ask a different question: does the Gaussian likelihood reproduce the overall distribution of reaction times?

### 7.2 Plot the observed and posterior-predictive ECDFs.

In [ ]:
azp.plot_ppc_dist(
    idata,
    var_names=["y"],
    kind="ecdf",
    figure_kwargs={"figsize": (7, 4)},
);

### 7.3 What would indicate adequate fit in this ECDF check?

The observed ECDF should be broadly consistent with the posterior-predictive ECDFs across the distribution rather than showing a persistent systematic displacement.

### 7.4 Does the Gaussian likelihood reproduce the overall distribution well?

Look especially at systematic discrepancies in the tails.

- answer here